### FLIGHT PRICE PREDICTION
- Based on the dataset, we are going to predict the price of a flight based on the features available.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
## reading the dataset and printing out a sample of 20 datapoints 
df=pd.read_excel('flight-price.xlsx')
df.sample(20)

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
6559,Air India,15/05/2019,Kolkata,Banglore,CCU → DEL → AMD → BLR,07:00,05:25 16 May,22h 25m,2 stops,No info,14424
9086,Multiple carriers,27/03/2019,Delhi,Cochin,DEL → BOM → COK,10:45,19:15,8h 30m,1 stop,No info,7154
203,Air India,18/05/2019,Delhi,Cochin,DEL → COK,05:10,08:00,2h 50m,non-stop,No info,6934
3650,Jet Airways,21/03/2019,Banglore,New Delhi,BLR → BOM → DEL,08:55,21:20,12h 25m,1 stop,In-flight meal not included,7832
3807,IndiGo,1/05/2019,Delhi,Cochin,DEL → BOM → COK,18:35,01:30 02 May,6h 55m,1 stop,No info,8854
4501,Jet Airways,24/03/2019,Kolkata,Banglore,CCU → BOM → BLR,06:30,16:20,9h 50m,1 stop,In-flight meal not included,8824
4757,Jet Airways,1/06/2019,Delhi,Cochin,DEL → BOM → COK,17:30,04:25 02 Jun,10h 55m,1 stop,No info,14714
4696,Jet Airways,1/04/2019,Kolkata,Banglore,CCU → BOM → BLR,18:55,08:15 02 Apr,13h 20m,1 stop,No info,12681
9078,SpiceJet,18/05/2019,Kolkata,Banglore,CCU → BLR,17:10,19:40,2h 30m,non-stop,No info,4174
7258,Multiple carriers,3/03/2019,Delhi,Cochin,DEL → BOM → COK,08:00,15:30,7h 30m,1 stop,No info,17057


In [3]:
#### Getting a rough view of your data information
#### on data types, number counts, missing values
df.info()
df.shape

<class 'pandas.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Airline          10683 non-null  str  
 1   Date_of_Journey  10683 non-null  str  
 2   Source           10683 non-null  str  
 3   Destination      10683 non-null  str  
 4   Route            10682 non-null  str  
 5   Dep_Time         10683 non-null  str  
 6   Arrival_Time     10683 non-null  str  
 7   Duration         10683 non-null  str  
 8   Total_Stops      10682 non-null  str  
 9   Additional_Info  10683 non-null  str  
 10  Price            10683 non-null  int64
dtypes: int64(1), str(10)
memory usage: 918.2 KB


(10683, 11)

#### Data Cleaning Process
- Our first insincts in to look at the data and see if there are any missing values and the data types and which of them are categorical and which are numerical. And based on the types do conversions necessary to fit the required data for training any machine learning model.


In [4]:
### DATA OF JOURNEY CLEANING
### We would do our best to make the data of journet seperated for a model to be able to understand. We could just convert to datetime but a model
### would not understand

df['Date_of_Journey']

0        24/03/2019
1         1/05/2019
2         9/06/2019
3        12/05/2019
4        01/03/2019
            ...    
10678     9/04/2019
10679    27/04/2019
10680    27/04/2019
10681    01/03/2019
10682     9/05/2019
Name: Date_of_Journey, Length: 10683, dtype: str

In [7]:
### We need to split Data into Day, Month, Year
# df["Date"]=df['Date_of_Journey'].str.split("/")
# df['Date'] ## -> [24, 03, 2019].......[9, 06, 2019]
# df.drop('Date')

### We need to split Data into Day, Month, Year
# df["Date"]=df['Date_of_Journey'].str.split("/")
# df['Date'] ## -> [24, 03, 2019].......[9, 06, 2019]

### Now we set the various values by indexing
df["Day"]=df['Date_of_Journey'].str.split("/").str[0]
df["Month"]=df['Date_of_Journey'].str.split("/").str[1]
df["Year"]=df['Date_of_Journey'].str.split("/").str[2]

### Convert to integers (models need numbers, not strings like "24")
df["Day"] = df["Day"].astype(int)
df["Month"] = df["Month"].astype(int)
df["Year"] = df["Year"].astype(int)

### Drop the original column — we already extracted what we need
### Use columns=... (or axis=1). Without that, pandas looks for a ROW named Date_of_Journey → KeyError
# df = df.drop(columns=["Date_of_Journey"])

df.head(10)



,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,Day,Month,Year
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897,24,3,2019
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662,1,5,2019
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882,9,6,2019
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218,12,5,2019
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302,1,3,2019
5,SpiceJet,24/06/2019,Kolkata,Banglore,CCU → BLR,09:00,11:25,2h 25m,non-stop,No info,3873,24,6,2019
6,Jet Airways,12/03/2019,Banglore,New Delhi,BLR → BOM → DEL,18:55,10:25 13 Mar,15h 30m,1 stop,In-flight meal not included,11087,12,3,2019
7,Jet Airways,01/03/2019,Banglore,New Delhi,BLR → BOM → DEL,08:00,05:05 02 Mar,21h 5m,1 stop,No info,22270,1,3,2019
8,Jet Airways,12/03/2019,Banglore,New Delhi,BLR → BOM → DEL,08:55,10:25 13 Mar,25h 30m,1 stop,In-flight meal not included,11087,12,3,2019
9,Multiple carriers,27/05/2019,Delhi,Cochin,DEL → BOM → COK,11:25,19:15,7h 50m,1 stop,No info,8625,27,5,2019


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Airline          10683 non-null  str  
 1   Source           10683 non-null  str  
 2   Destination      10683 non-null  str  
 3   Route            10682 non-null  str  
 4   Dep_Time         10683 non-null  str  
 5   Arrival_Time     10683 non-null  str  
 6   Duration         10683 non-null  str  
 7   Total_Stops      10682 non-null  str  
 8   Additional_Info  10683 non-null  str  
 9   Price            10683 non-null  int64
 10  Day              10683 non-null  int64
 11  Month            10683 non-null  int64
 12  Year             10683 non-null  int64
dtypes: int64(4), str(9)
memory usage: 1.1 MB


In [ ]:
# ### Drop the df['Date_of_Journey'] axis since its no longer required
df.drop('Date_of_Journey', axis=1, inplace=True)
df.head()

,Airline,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,Day,Month,Year,Arrival_Hour
0,IndiGo,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897,24,3,2019,1
1,Air India,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662,1,5,2019,13
2,Jet Airways,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882,9,6,2019,4
3,IndiGo,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218,12,5,2019,23
4,IndiGo,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302,1,3,2019,21


In [19]:
#### We move to Arrival Time and split by column ":" to get the Hour and Minutes
## but unfortunate some of the datapoints have the format 23:56 and others 04:45 22 Jun
## Hence we need to get rid of the extra data on some datapoints to get just hour and minutes

df['Arrival_Hour']=df['Arrival_Time'].str.split(":").str[0].astype(int)

In [24]:
#### Now we need to grab minutes and get rid of the minutes with dates Hour:Minute Date -> 01:10 22 Mar
df['Arrival_Minute']=df['Arrival_Time'].str.split(":").str[1].str.split(" ").str[0].astype(int)

In [25]:
df.head(10)

,Airline,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,Day,Month,Year,Arrival_Hour,Arrival_Minute
0,IndiGo,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897,24,3,2019,1,10
1,Air India,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662,1,5,2019,13,15
2,Jet Airways,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882,9,6,2019,4,25
3,IndiGo,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218,12,5,2019,23,30
4,IndiGo,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302,1,3,2019,21,35
5,SpiceJet,Kolkata,Banglore,CCU → BLR,09:00,11:25,2h 25m,non-stop,No info,3873,24,6,2019,11,25
6,Jet Airways,Banglore,New Delhi,BLR → BOM → DEL,18:55,10:25 13 Mar,15h 30m,1 stop,In-flight meal not included,11087,12,3,2019,10,25
7,Jet Airways,Banglore,New Delhi,BLR → BOM → DEL,08:00,05:05 02 Mar,21h 5m,1 stop,No info,22270,1,3,2019,5,5
8,Jet Airways,Banglore,New Delhi,BLR → BOM → DEL,08:55,10:25 13 Mar,25h 30m,1 stop,In-flight meal not included,11087,12,3,2019,10,25
9,Multiple carriers,Delhi,Cochin,DEL → BOM → COK,11:25,19:15,7h 50m,1 stop,No info,8625,27,5,2019,19,15


In [ ]:
## Lets drop the Arrival Time now
df.drop('Arrival_Time', axis=1, inplace=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Airline          10683 non-null  str  
 1   Source           10683 non-null  str  
 2   Destination      10683 non-null  str  
 3   Route            10682 non-null  str  
 4   Dep_Time         10683 non-null  str  
 5   Duration         10683 non-null  str  
 6   Total_Stops      10682 non-null  str  
 7   Additional_Info  10683 non-null  str  
 8   Price            10683 non-null  int64
 9   Day              10683 non-null  int64
 10  Month            10683 non-null  int64
 11  Year             10683 non-null  int64
 12  Arrival_Hour     10683 non-null  int64
 13  Arrival_Minute   10683 non-null  int64
dtypes: int64(6), str(8)
memory usage: 1.1 MB


In [32]:
#### We move to Departure Time and split by column ":" to get the Hour and Minutes
## Dep_Time is usually in the format HH:MM (e.g. 22:20), but we follow the same approach as Arrival
## Hence we extract the hour and convert it to int for the model

df['Departure_Hour']=df['Dep_Time'].str.split(":").str[0].astype(int)
df['Departure_Minute']=df['Dep_Time'].str.split(":").str[1].str.split(" ").str[0].astype(int)
df.sample(10)

,Airline,Source,Destination,Route,Dep_Time,Duration,Total_Stops,Additional_Info,Price,Day,Month,Year,Arrival_Hour,Arrival_Minute,Departure_Hour,Departure_Minute
8186,Multiple carriers,Delhi,Cochin,DEL → BOM → COK,10:20,10h 40m,1 stop,No info,6637,27,3,2019,21,0,10,20
8843,IndiGo,Mumbai,Hyderabad,BOM → HYD,02:35,1h 30m,non-stop,No info,2754,15,5,2019,4,5,2,35
6955,Air India,Delhi,Cochin,DEL → BOM → COK,10:00,9h 15m,1 stop,No info,8372,9,5,2019,19,15,10,0
1169,IndiGo,Delhi,Cochin,DEL → LKO → COK,21:50,5h 45m,1 stop,No info,6195,6,6,2019,3,35,21,50
1671,IndiGo,Chennai,Kolkata,MAA → CCU,13:15,2h 20m,non-stop,No info,3597,12,6,2019,15,35,13,15
6288,Multiple carriers,Delhi,Cochin,DEL → HYD → COK,09:45,12h 45m,1 stop,No info,9646,1,6,2019,22,30,9,45
1973,Air India,Banglore,New Delhi,BLR → BOM → BHO → DEL,08:50,24h 35m,2 stops,No info,12725,12,3,2019,9,25,8,50
1414,Jet Airways,Banglore,Delhi,BLR → DEL,18:55,3h 5m,non-stop,In-flight meal not included,4030,1,5,2019,22,0,18,55
7718,Air Asia,Banglore,Delhi,BLR → DEL,23:55,2h 50m,non-stop,No info,4483,1,5,2019,2,45,23,55
803,Air India,Banglore,New Delhi,BLR → CCU → GAU → DEL,05:50,16h 20m,2 stops,No info,9796,15,3,2019,22,10,5,50


In [33]:
df.drop('Dep_Time', axis=1, inplace=True)

#### Duration — Thought Process (before cleaning)

`Duration` looks similar to Arrival/Dep time (strings like `2h 50m`, `19h`), but it means something different.

**Arrival / Departure** answer: *when* does the flight leave or land?  
→ Split into **Hour** and **Minute** (time-of-day features).

**Duration** answers: *how long* is the trip?  
→ Convert to **one numeric length**, typically **total minutes**.

##### Why total minutes (not Hour + Minute split)?

1. Duration is a **magnitude** (trip length), not a clock time. `2h 50m` and `19h` should be ordered on one scale.
2. Splitting into `Duration_Hour` + `Duration_Minute` is possible but usually **weaker / redundant** — the model mainly needs total length.
3. Leaving it as a string (`"2h 50m"`) is bad for modeling — the model cannot treat longer flights as “larger” unless we encode it numerically.

##### Minutes vs hours?

- **Total minutes (int)** — most common (e.g. `2h 50m` → `170`)
- Hours as a float also works (`2.83...`), but minutes keep values as clean integers

##### Formats we must handle later (when we clean)

| Example | Meaning |
|---|---|
| `2h 50m` | hours + minutes (most rows) |
| `19h` / `24h` | hours only |
| rare `…m` only | minutes only |

**Decision for this notebook:** convert `Duration` → **`Duration_mins`** (total minutes), then drop the original string column — same cleanup idea as dropping `Arrival_Time` after extracting hour/minute.

*(Cleaning code comes next — not done in this cell.)*


In [38]:
#### Convert Duration to total minutes
## Formats we see: "2h 50m" (hours + minutes), "19h" (hours only), rare minutes-only
## Plan: extract hours and minutes separately → fill missing with 0 → Duration_mins = hours*60 + minutes

## Pull the number before "h" (e.g. "2h 50m" → 2, "19h" → 19). If no hours, becomes NaN
df['hours'] = df['Duration'].str.extract(r'(\d+)h')


## Pull the number before "m" (e.g. "2h 50m" → 50). If no minutes (like "19h"), becomes NaN
df['mins'] = df['Duration'].str.extract(r'(\d+)m')

## Missing hours/minutes mean 0 (e.g. "19h" has 0 minutes; rare "5m" has 0 hours)
df['hours'] = df['hours'].fillna(0).astype(int)
df['mins'] = df['mins'].fillna(0).astype(int)

## One numeric feature the model can use as trip length
df['Duration_mins'] = df['hours'] * 60 + df['mins']

## Drop helpers + original string column (same idea as dropping Arrival_Time / Dep_Time)
df.drop(['Duration', 'hours', 'mins'], axis=1, inplace=True)



In [39]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Airline           10683 non-null  str  
 1   Source            10683 non-null  str  
 2   Destination       10683 non-null  str  
 3   Route             10682 non-null  str  
 4   Total_Stops       10682 non-null  str  
 5   Additional_Info   10683 non-null  str  
 6   Price             10683 non-null  int64
 7   Day               10683 non-null  int64
 8   Month             10683 non-null  int64
 9   Year              10683 non-null  int64
 10  Arrival_Hour      10683 non-null  int64
 11  Arrival_Minute    10683 non-null  int64
 12  Departure_Hour    10683 non-null  int64
 13  Departure_Minute  10683 non-null  int64
 14  Duration_mins     10683 non-null  int64
dtypes: int64(9), str(6)
memory usage: 1.2 MB


#### Total_Stops — Thought Process (before cleaning)

`Total_Stops` is a **categorical** string column (`non-stop`, `1 stop`, `2 stops`, …), but it is not like `Airline` or `Source`.

##### What kind of categorical is it?

| Type | Meaning | Example in this dataset |
|---|---|---|
| **Nominal** | No natural order | Airline, Source, Destination |
| **Ordinal** | Categories have a meaningful order | **Total_Stops** (`0 < 1 < 2 < 3 < 4`) |

More stops clearly means “more.” Price and duration often increase with stops, so that order matters for the model.

##### Typical approach: ordinal map → integers

```text
non-stop → 0
1 stop   → 1
2 stops  → 2
3 stops  → 3
4 stops  → 4
```

##### Why this choice (not one-hot)?

1. **Natural order exists** — the model should treat more stops as a larger value on one scale.
2. **One-hot ignores order** — separate dummy columns treat `non-stop` and `4 stops` as unrelated labels; you lose the “more stops → higher” signal unless the model relearns it.
3. **Low cardinality + ordered** — only a few levels, already numeric in meaning → one `int` column is simple and strong.
4. **Different from Airline/Source** — those are nominal → usually one-hot (or similar). `Total_Stops` is ordinal → **map to stop count**.

##### Also note before coding

- There is **1 missing value** (`NaN`) — handle it when implementing (inspect the row, then drop or fill).

**Decision for this notebook:** encode `Total_Stops` as an integer stop count (0–4).

*(Cleaning / mapping code comes next — not done in this cell.)*


In [43]:
### Now lets take alook at Total stops a categorical variable
df["Total_Stops"].unique()

<StringArray>
['non-stop', '2 stops', '1 stop', '3 stops', nan, '4 stops']
Length: 6, dtype: str

In [44]:
### Check the number of nan values 
df[df["Total_Stops"].isnull()]

,Airline,Source,Destination,Route,Total_Stops,Additional_Info,Price,Day,Month,Year,Arrival_Hour,Arrival_Minute,Departure_Hour,Departure_Minute,Duration_mins
9039,Air India,Delhi,Cochin,NaN,NaN,No info,7480,6,5,2019,9,25,9,45,1420


In [45]:
### We need to map to stop counts
df['Total_Stops'] = df['Total_Stops'].map({
    'non-stop': 0,
    '1 stop': 1,
    '2 stops': 2,
    '3 stops': 3,
    '4 stops': 4,
    np.nan:1
})

#### Route — Why We Typically Drop It

`Route` looks informative (`DEL → BOM → COK`, `BLR → DEL`, …), but in a first feature-engineering pass we usually **drop** it.

##### 1. It mostly duplicates features we already keep

A route already implies:
- start ≈ `Source`
- end ≈ `Destination`
- how many hops ≈ `Total_Stops`

Example: `DEL → BOM → COK` lines up with Source Delhi, Destination Cochin, and 2 stops.  
So much of Route’s signal is already captured more cleanly elsewhere.

##### 2. High cardinality

There are **~128 unique routes**, vs only a handful of Source / Destination / Total_Stops values.

One-hot encoding Route would create many sparse columns → noisy, memory-heavy, and easy to **overfit** (rare routes appear only a few times).

##### 3. Messy string / not model-friendly as-is

Values like `CCU → IXR → BBI → BLR` are long path strings. Using them well needs extra engineering (split airports, count stops from the path, etc.). For a standard pipeline, that cost often outweighs the benefit.

##### Decision for this notebook

| Keep | Drop (for now) |
|---|---|
| `Source`, `Destination`, `Total_Stops` | **`Route`** |

**Rule of thumb:** keep the simpler features that already explain origin, destination, and stops; drop the raw Route string.

*(If we want more signal later, we can engineer from Route — e.g. number of airports — instead of keeping the raw categorical path.)*


In [48]:
df.sample(10)

,Airline,Source,Destination,Route,Total_Stops,Additional_Info,Price,Day,Month,Year,Arrival_Hour,Arrival_Minute,Departure_Hour,Departure_Minute,Duration_mins
5530,Jet Airways,Banglore,New Delhi,BLR → BOM → DEL,1,In-flight meal not included,7832,21,3,2019,20,20,11,40,520
6576,Jet Airways,Banglore,New Delhi,BLR → BOM → DEL,1,1 Long layover,31825,1,3,2019,9,30,18,40,890
1610,Jet Airways,Kolkata,Banglore,CCU → BOM → BLR,1,In-flight meal not included,10844,24,5,2019,22,35,20,0,1595
8031,Jet Airways,Banglore,Delhi,BLR → DEL,0,In-flight meal not included,3502,9,5,2019,22,50,19,50,180
5756,IndiGo,Delhi,Cochin,DEL → HYD → COK,1,No info,6287,15,4,2019,12,10,5,5,425
2075,Jet Airways,Kolkata,Banglore,CCU → BOM → BLR,1,In-flight meal not included,9899,12,6,2019,18,15,8,25,590
9014,IndiGo,Mumbai,Hyderabad,BOM → HYD,0,No info,4049,27,6,2019,7,55,6,25,90
8174,Jet Airways,Delhi,Cochin,DEL → AMD → BOM → COK,2,In-flight meal not included,6643,21,3,2019,18,50,19,10,1420
10301,Jet Airways,Delhi,Cochin,DEL → BOM → COK,1,In-flight meal not included,10262,27,6,2019,12,35,15,0,1295
10169,Jet Airways,Delhi,Cochin,DEL → BOM → COK,1,In-flight meal not included,10262,15,6,2019,12,35,18,15,1100


In [ ]:
### For Route we could drop it because we have Destination and Source and total_stops
df.drop('Route', axis=1, inplace=True)
df.sample(10)

,Airline,Source,Destination,Total_Stops,Additional_Info,Price,Day,Month,Year,Arrival_Hour,Arrival_Minute,Departure_Hour,Departure_Minute,Duration_mins
7632,IndiGo,Delhi,Cochin,1,No info,5054,24,6,2019,12,10,7,35,275
7315,IndiGo,Delhi,Cochin,1,No info,5298,15,6,2019,12,10,6,50,320
10018,Jet Airways,Delhi,Cochin,1,In-flight meal not included,10577,9,6,2019,19,0,19,45,1395
146,Jet Airways,Delhi,Cochin,2,In-flight meal not included,15318,3,6,2019,4,25,20,0,505
6714,Jet Airways,Mumbai,Hyderabad,0,No info,8040,1,5,2019,11,50,10,20,90
8423,Jet Airways,Delhi,Cochin,1,In-flight meal not included,11399,9,5,2019,19,0,8,0,660
1101,Jet Airways,Kolkata,Banglore,1,No info,14178,24,5,2019,12,0,6,30,330
2103,Multiple carriers,Delhi,Cochin,1,No info,12717,24,6,2019,19,0,10,0,540
2410,Jet Airways,Banglore,New Delhi,1,No info,16736,6,3,2019,7,40,20,35,665
1717,IndiGo,Kolkata,Banglore,0,No info,6565,1,3,2019,17,45,15,10,155


#### Airline, Source, Destination — Decision Making

These three are **nominal categorical** features: labels with **no natural order** (unlike `Total_Stops`).

| Feature | Approx. unique values | Keep or drop? | Encoding |
|---|---|---|---|
| **Airline** | ~12 | **Keep** | One-hot |
| **Source** | 5 | **Keep** | One-hot |
| **Destination** | 6 | **Keep** | One-hot |
| Total_Stops | 5 | Keep | Ordinal map (0–4) |
| Route | ~128 | Drop | — |

##### Why keep them?

Airline brand and city pair strongly affect ticket **Price**. Dropping them would throw away useful signal. They are not redundant like `Route` (which we already cover with Source, Destination, and Total_Stops).

##### Why one-hot encode (not ordinal map)?

1. **No ranking** — IndiGo is not “less than” Jet Airways; Delhi is not “greater than” Mumbai. Mapping to 1, 2, 3 would invent a fake order.
2. **Low / moderate cardinality** — only 5–12 levels, so one-hot is cheap and standard.
3. **What one-hot does** — each category becomes its own 0/1 column, e.g. `Airline_IndiGo=1`, `Source_Delhi=1`.

```python
# Typical implementation later:
pd.get_dummies(df, columns=['Airline', 'Source', 'Destination'])
# optional: drop_first=True for linear models (avoids dummy-variable trap)
```

##### Contrast with other categoricals in this notebook

| Feature type | Example | Typical choice |
|---|---|---|
| **Nominal** | Airline, Source, Destination | **One-hot** |
| **Ordinal** | Total_Stops | Map to integers |
| **High-cardinality / redundant** | Route | Drop |

##### Small caveats (for later EDA / cleaning)

- **Airline** has a few rare labels (`Trujet`, some premium/business variants) — first pass: one-hot; later you may group rares as `"Other"`.
- **Destination** has both `Delhi` and `New Delhi` — often kept separate (or merged after checking); still one-hot either way.

**Decision for this notebook:** keep `Airline`, `Source`, and `Destination` → encode with **one-hot (`get_dummies`)**.

*(Encoding code comes next — not done in this cell.)*


In [ ]:
### Additional Info


<StringArray>
[                     'No info',  'In-flight meal not included',
 'No check-in baggage included',              '1 Short layover',
                      'No Info',               '1 Long layover',
              'Change airports',               'Business class',
               'Red-eye flight',               '2 Long layover']
Length: 10, dtype: str